# AI 與人工評分比較

支援三種模式：

- `MODE = "ai"`：從 RAG eval CSV 抽樣、AI 評分，建立新的 `rag_ai_manual_eval_results/YYYYMMDD/vN/`。
- `MODE = "compare"`：讀取某次 run 的 AI 評分檔，加上人工 `human_score` 後比較，並輸出 compare CSV 與 Markdown 報告。
- `MODE = "read"`：讀取某次 run 的 compare 結果與報告，方便檢視內容。

In [ ]:
from pathlib import Path

# ai / compare / read
MODE = "read"

# MODE = "ai" 時使用：指定現在的 RAG eval 檔案
MAIN_EVAL_CSV = Path("eval_ragas_testset/20260603/v1/ragas_eval_with_response.csv")
SAMPLE_SIZE = 10
RANDOM_SEED = 42
MODEL_NAME = "gpt-4o-mini"
SLEEP_SEC = 0

# MODE = "compare" / "read" 時使用：指定要操作哪一次 manual eval run
RESULT_ROOT = Path("rag_ai_manual_eval_results")
TARGET_DATE = "20260522"
TARGET_VERSION = "v1"

# MODE = "compare" 時使用：
# 如果 human_score 已經填在 ai_scored_10.csv，保持 None。
# 如果人工評分另存一份 CSV，填檔名，例如 "human_ai_scored_10.csv"。
HUMAN_SCORED_FILENAME = None

# MODE = "read" 時使用：None 表示看全部；也可填 "Q1" 只看單題。
READ_SAMPLE_ID = None

In [ ]:
import json
import os
import re
import time
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

SYSTEM_PROMPT = """
你是一位協助研究者評估 RAG 系統回答品質的嚴謹評分員。
請根據使用者問題 user_input、RAG 回答 response、檢索內容 retrieved_contexts 判斷回答品質。

主要判斷：
1. 回答是否有回應使用者問題。
2. 回答主要內容是否能由 retrieved_contexts 支持。
3. 回答是否包含 retrieved_contexts 無法支持的臆測、錯誤或過度推論。

評分標準：
1 分：回答與問題無關，或無法由 retrieved_contexts 支持。
2 分：只回答少部分問題，或大多內容缺乏 retrieved_contexts 支持。
3 分：大致回答問題，但內容不完整，或部分敘述無法由 retrieved_contexts 明確支持。
4 分：回答有回應問題，且大部分內容可由 retrieved_contexts 支持，只有輕微不足。
5 分：完整回答問題，主要內容皆可由 retrieved_contexts 明確支持，沒有明顯錯誤或幻覺。

請只根據提供內容評分，不要自行補充外部知識。請輸出 JSON。
""".strip()


def score_to_binary(score):
    try:
        score = int(score)
    except Exception:
        return None
    if score in [1, 2, 3]:
        return "未達可接受回答"
    if score in [4, 5]:
        return "達可接受回答"
    return None


def version_number(path):
    match = re.fullmatch(r"v(\d+)", path.name)
    return int(match.group(1)) if match else None


def create_new_run_dir(result_root=RESULT_ROOT):
    date_dir = Path(result_root) / datetime.now().strftime("%Y%m%d")
    date_dir.mkdir(parents=True, exist_ok=True)
    versions = [version_number(p) for p in date_dir.iterdir() if p.is_dir()]
    versions = [v for v in versions if v is not None]
    run_dir = date_dir / f"v{max(versions, default=0) + 1}"
    run_dir.mkdir()
    return run_dir


def get_run_dir(result_root=RESULT_ROOT, target_date=TARGET_DATE, target_version=TARGET_VERSION):
    run_dir = Path(result_root) / target_date / target_version
    if not run_dir.exists():
        raise FileNotFoundError(f"找不到指定資料夾：{run_dir}")
    return run_dir


def output_paths(run_dir, sample_size=SAMPLE_SIZE):
    return {
        "sample_csv": run_dir / f"rag_sample_{sample_size}.csv",
        "ai_scored_csv": run_dir / f"ai_scored_{sample_size}.csv",
        "compare_csv": run_dir / f"compare_result_{sample_size}.csv",
        "metrics_csv": run_dir / f"compare_result_{sample_size}_metrics.csv",
        "report_md": run_dir / "evaluation_report.md",
        "run_info_json": run_dir / "run_info.json",
        "prompt_txt": run_dir / "prompt_used.txt",
    }


def find_existing_file(run_dir, prefix, suffix=".csv"):
    matches = sorted(run_dir.glob(f"{prefix}*{suffix}"))
    if not matches:
        raise FileNotFoundError(f"{run_dir} 找不到 {prefix}*{suffix}")
    return matches[0]


def sample_from_main_eval(input_csv, output_csv, sample_size=SAMPLE_SIZE, random_seed=RANDOM_SEED):
    df = pd.read_csv(input_csv, encoding="utf-8-sig")
    required_cols = ["user_input", "response", "retrieved_contexts"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"缺少必要欄位：{missing}")

    if "status" in df.columns:
        success_df = df[df["status"].isin(["success", "completed", "ok", "SUCCESS", "Completed", "OK"])]
        df_for_sample = success_df if len(success_df) >= sample_size else df
    else:
        df_for_sample = df

    if len(df_for_sample) < sample_size:
        raise ValueError(f"可抽樣資料只有 {len(df_for_sample)} 筆，小於 SAMPLE_SIZE={sample_size}")

    sample_df = df_for_sample.sample(n=sample_size, random_state=random_seed).copy()
    sample_df.insert(0, "sample_id", [f"Q{i + 1}" for i in range(len(sample_df))])
    sample_df.insert(1, "original_index", sample_df.index)
    sample_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    return sample_df


def get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError("找不到 OPENAI_API_KEY，請先設定環境變數或 .env。")
    return OpenAI(api_key=api_key)


def evaluate_one_row(row, model=MODEL_NAME):
    client = get_openai_client()
    user_prompt = f"""
【user_input】
{row.get('user_input', '')}

【response】
{row.get('response', '')}

【retrieved_contexts】
{row.get('retrieved_contexts', '')}

請給出 1 到 5 分，並簡短說明原因。
""".strip()

    response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "rag_answer_score",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "score": {"type": "integer", "minimum": 1, "maximum": 5},
                    "reason": {"type": "string"},
                },
                "required": ["score", "reason"],
                "additionalProperties": False,
            },
        },
    }
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}],
        response_format=response_format,
        temperature=0,
    )
    result = json.loads(completion.choices[0].message.content)
    return result["score"], result["reason"]


def run_ai_scoring(input_csv, output_csv, model=MODEL_NAME, sleep_sec=SLEEP_SEC):
    df = pd.read_csv(input_csv, encoding="utf-8-sig")
    ai_scores = []
    ai_reasons = []
    for idx, row in df.iterrows():
        sample_id = row.get("sample_id", f"Q{idx + 1}")
        print(f"AI 評分 {idx + 1}/{len(df)}：{sample_id}")
        try:
            score, reason = evaluate_one_row(row, model=model)
        except Exception as e:
            score, reason = None, f"AI 評分失敗：{e}"
        ai_scores.append(score)
        ai_reasons.append(reason)
        time.sleep(sleep_sec)

    df["ai_score"] = ai_scores
    df["ai_reason"] = ai_reasons
    df["ai_binary"] = df["ai_score"].apply(score_to_binary)
    df["human_score"] = ""
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    return df


def distribution_table(df, score_column):
    numeric = pd.to_numeric(df[score_column], errors="coerce")
    return numeric.value_counts().reindex([1, 2, 3, 4, 5], fill_value=0).rename_axis("score").reset_index(name="count")


def binary_distribution_table(df, binary_column):
    return df[binary_column].value_counts().reindex(["未達可接受回答", "達可接受回答"], fill_value=0).rename_axis("binary_label").reset_index(name="count")


def add_count_columns(df):
    for actor in ["ai", "human"]:
        score_col = f"{actor}_score"
        binary_col = f"{actor}_binary"
        if score_col in df.columns:
            score_counts = pd.to_numeric(df[score_col], errors="coerce").value_counts(dropna=False)
            df[f"{actor}_score_count"] = pd.to_numeric(df[score_col], errors="coerce").map(score_counts).fillna(0).astype(int)
        if binary_col in df.columns:
            binary_counts = df[binary_col].value_counts(dropna=False)
            df[f"{actor}_binary_count"] = df[binary_col].map(binary_counts).fillna(0).astype(int)
    return df


def compare_ai_human(input_csv, output_csv, metrics_output_csv, report_md):
    df = pd.read_csv(input_csv, encoding="utf-8-sig")
    missing = [col for col in ["ai_score", "human_score"] if col not in df.columns]
    if missing:
        raise ValueError(f"缺少必要欄位：{missing}。請先在 AI 評分檔加入 human_score。")

    df["ai_score"] = pd.to_numeric(df["ai_score"], errors="coerce")
    df["human_score"] = pd.to_numeric(df["human_score"], errors="coerce")
    df["ai_binary"] = df["ai_score"].apply(score_to_binary)
    df["human_binary"] = df["human_score"].apply(score_to_binary)
    valid_df = df.dropna(subset=["ai_binary", "human_binary"]).copy()

    positive = "達可接受回答"
    negative = "未達可接受回答"
    tp = len(valid_df[(valid_df["human_binary"] == positive) & (valid_df["ai_binary"] == positive)])
    fp = len(valid_df[(valid_df["human_binary"] == negative) & (valid_df["ai_binary"] == positive)])
    fn = len(valid_df[(valid_df["human_binary"] == positive) & (valid_df["ai_binary"] == negative)])
    tn = len(valid_df[(valid_df["human_binary"] == negative) & (valid_df["ai_binary"] == negative)])
    total = tp + fp + fn + tn

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    accuracy = (tp + tn) / total if total else 0
    exact = (valid_df["ai_score"] == valid_df["human_score"]).mean() if len(valid_df) else 0
    binary_agreement = (valid_df["ai_binary"] == valid_df["human_binary"]).mean() if len(valid_df) else 0

    metrics = pd.DataFrame([{
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy,
        "ExactScoreAgreement": exact,
        "BinaryAgreement": binary_agreement,
        "ValidN": len(valid_df),
    }])

    valid_df = add_count_columns(valid_df)
    valid_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    metrics.to_csv(metrics_output_csv, index=False, encoding="utf-8-sig")

    write_report(report_md, valid_df, metrics)
    return valid_df, metrics


def markdown_table(df):
    cols = list(df.columns)
    lines = [
        "| " + " | ".join(map(str, cols)) + " |",
        "| " + " | ".join(["---"] * len(cols)) + " |",
    ]
    for _, row in df.iterrows():
        values = [str(row[col]).replace("|", "\\|") for col in cols]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


def write_report(report_md, compare_df, metrics):
    ai_score_dist = distribution_table(compare_df, "ai_score")
    human_score_dist = distribution_table(compare_df, "human_score")
    ai_binary_dist = binary_distribution_table(compare_df, "ai_binary")
    human_binary_dist = binary_distribution_table(compare_df, "human_binary")
    m = metrics.iloc[0]

    rows = []
    for _, row in compare_df.iterrows():
        rows.append(
            f"### {row.get('sample_id', '')}\n\n"
            f"**問題**\n\n{row.get('user_input', '')}\n\n"
            f"**RAG 回答**\n\n{row.get('response', '')}\n\n"
            f"**AI 分數**：{row.get('ai_score', '')} / {row.get('ai_binary', '')}\n\n"
            f"**人工分數**：{row.get('human_score', '')} / {row.get('human_binary', '')}\n\n"
            f"**AI 理由**\n\n{row.get('ai_reason', '')}\n"
        )

    report = f"""
# RAG AI / Human Evaluation Report

Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Summary

| Metric | Value |
|---|---:|
| Valid N | {int(m['ValidN'])} |
| Precision | {m['Precision']:.4f} |
| Recall | {m['Recall']:.4f} |
| F1 | {m['F1']:.4f} |
| Accuracy | {m['Accuracy']:.4f} |
| Exact score agreement | {m['ExactScoreAgreement']:.4f} |
| Binary agreement | {m['BinaryAgreement']:.4f} |

## Confusion Matrix

| Human \\ AI | 達可接受回答 | 未達可接受回答 |
|---|---:|---:|
| 達可接受回答 | {int(m['TP'])} | {int(m['FN'])} |
| 未達可接受回答 | {int(m['FP'])} | {int(m['TN'])} |

## AI Score Distribution

{markdown_table(ai_score_dist)}

## Human Score Distribution

{markdown_table(human_score_dist)}

## AI Binary Distribution

{markdown_table(ai_binary_dist)}

## Human Binary Distribution

{markdown_table(human_binary_dist)}

## Samples

{chr(10).join(rows)}
""".strip()

    report_md.write_text(report, encoding="utf-8")


def save_run_info(run_dir, paths, extra=None):
    info = {
        "mode": MODE,
        "main_eval_csv": str(MAIN_EVAL_CSV),
        "sample_size": SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
        "model_name": MODEL_NAME,
        "output_dir": str(run_dir),
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    if extra:
        info.update(extra)
    paths["run_info_json"].write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["prompt_txt"].write_text(SYSTEM_PROMPT, encoding="utf-8")


def run_ai_mode():
    run_dir = create_new_run_dir()
    paths = output_paths(run_dir)
    print(f"輸出資料夾：{run_dir}")
    sample_df = sample_from_main_eval(MAIN_EVAL_CSV, paths["sample_csv"])
    scored_df = run_ai_scoring(paths["sample_csv"], paths["ai_scored_csv"])
    save_run_info(run_dir, paths, {"sample_csv": str(paths["sample_csv"]), "ai_scored_csv": str(paths["ai_scored_csv"])})
    display(scored_df.head())
    print(f"請在 {paths['ai_scored_csv']} 填入 human_score 後，再用 MODE='compare'。")


def run_compare_mode():
    run_dir = get_run_dir()
    paths = output_paths(run_dir)
    input_csv = run_dir / HUMAN_SCORED_FILENAME if HUMAN_SCORED_FILENAME else find_existing_file(run_dir, "ai_scored_")
    compare_df, metrics = compare_ai_human(input_csv, paths["compare_csv"], paths["metrics_csv"], paths["report_md"])
    save_run_info(run_dir, paths, {"compare_input_csv": str(input_csv), "compare_csv": str(paths["compare_csv"]), "report_md": str(paths["report_md"])})
    display(metrics)
    display(compare_df[["sample_id", "ai_score", "human_score", "ai_binary", "human_binary", "ai_score_count", "human_score_count"]])
    print(f"已輸出 compare：{paths['compare_csv']}")
    print(f"已輸出報告：{paths['report_md']}")


def run_read_mode():
    run_dir = get_run_dir()
    compare_csv = find_existing_file(run_dir, "compare_result_")
    report_md = run_dir / "evaluation_report.md"
    df = pd.read_csv(compare_csv, encoding="utf-8-sig")
    if READ_SAMPLE_ID:
        df = df[df["sample_id"].astype(str) == str(READ_SAMPLE_ID)]
    print(f"讀取：{compare_csv}")
    display(df)
    if report_md.exists() and not READ_SAMPLE_ID:
        display(Markdown(report_md.read_text(encoding="utf-8")))


if MODE == "ai":
    run_ai_mode()
elif MODE == "compare":
    run_compare_mode()
elif MODE == "read":
    run_read_mode()
else:
    raise ValueError("MODE 只能是 'ai', 'compare', 'read'")